In [2]:
# Import the modules
import pandas as pd
import hvplot.pandas
from pathlib import Path
from sklearn.cluster import KMeans

### Read in the CSV file and prepare the Pandas DataFrame

In [3]:
# Read the csv file into a pandas DataFrame
customers_transformed_df = pd.read_csv(
    Path("resources/cleaned_data/crime_2020.csv")
)

# Review the DataFrame
customers_transformed_df.head()

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON,crime_timestamp,Year
0,190326475,03/01/2020 12:00:00 AM,2020-03-01,2130,7,Wilshire,784,1,510,VEHICLE - STOLEN,...,510.0,998.0,NaN,NaN,1900 S LONGWOOD AV,NaN,34.0375,-118.3506,2020-03-01 21:30:00,2020
1,200106753,02/09/2020 12:00:00 AM,2020-02-08,1800,1,Central,182,1,330,BURGLARY FROM VEHICLE,...,330.0,998.0,NaN,NaN,1000 S FLOWER ST,NaN,34.0444,-118.2628,2020-02-08 18:00:00,2020
2,200320258,11/11/2020 12:00:00 AM,2020-11-04,1700,3,Southwest,356,1,480,BIKE - STOLEN,...,480.0,NaN,NaN,NaN,1400 W 37TH ST,NaN,34.0210,-118.3002,2020-11-04 17:00:00,2020
3,200907217,05/10/2023 12:00:00 AM,2020-03-10,2037,9,Van Nuys,964,1,343,SHOPLIFTING-GRAND THEFT ($950.01 & OVER),...,343.0,NaN,NaN,NaN,14000 RIVERSIDE DR,NaN,34.1576,-118.4387,2020-03-10 20:37:00,2020
4,200412582,09/09/2020 12:00:00 AM,2020-09-09,630,4,Hollenbeck,413,1,510,VEHICLE - STOLEN,...,510.0,NaN,NaN,NaN,200 E AVENUE 28,NaN,34.0820,-118.2130,2020-09-09 06:30:00,2020


In [4]:
customers_transformed_df = customers_transformed_df[['TIME OCC', 'AREA', 'Rpt Dist No', 'LAT', 'LON']] #'AREA', 'Rpt Dist No', 'Part 1-2'
customers_transformed_df.head()

,TIME OCC,AREA,Rpt Dist No,LAT,LON
0,2130,7,784,34.0375,-118.3506
1,1800,1,182,34.0444,-118.2628
2,1700,3,356,34.0210,-118.3002
3,2037,9,964,34.1576,-118.4387
4,630,4,413,34.0820,-118.2130


In [5]:
# customers_transformed_df = customers_transformed_df.iloc[0:200]
customers_transformed_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199805 entries, 0 to 199804
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   TIME OCC     199805 non-null  int64  
 1   AREA         199805 non-null  int64  
 2   Rpt Dist No  199805 non-null  int64  
 3   LAT          199805 non-null  float64
 4   LON          199805 non-null  float64
dtypes: float64(2), int64(3)
memory usage: 7.6 MB


### Step 1: Use PCA to reduce the dimensionality of the transformed customers DataFrame to 2 principal components

In [6]:
# Import the PCA module
from sklearn.decomposition import PCA

In [7]:
# Instantiate the PCA instance and declare the number of PCA variables
pca=PCA(n_components=2)

In [8]:
# Fit the PCA model on the transformed credit card DataFrame
customers_pca = pca.fit_transform(customers_transformed_df)

# Review the first 5 rows of the array of list data
customers_pca[:5]

array([[-761.26313292, -382.57782577],
       [-402.95233572, -968.20003936],
       [-311.35723713, -789.62157128],
       [-676.94596608, -198.33999431],
       [ 754.71115359, -681.69564948]])

### Step 2: Using the explained_variance_ratio_ function from PCA, calculate the percentage of the total variance that is captured by the two PCA variables.

In [9]:
# Calculate the PCA explained variance ratio
pca.explained_variance_ratio_

array([0.53613009, 0.46379371])

**Question:** What is the explained variance ratio captured by the two PCA variables?
    
**Answer:** About 85% of the total variance is condensed into the 2 PCA variables.

### Step 3: Using the customer_pca data, create a Pandas DataFrame called customers_pca_df. The columns of the DataFrame should be called "PCA1" and "PCA2".

In [10]:
# Create the PCA DataFrame
customers_pca_df = pd.DataFrame(
    customers_pca,
    columns=["PCA1", "PCA2"]
)

# Review the PCA DataFrame
customers_pca_df.head()

,PCA1,PCA2
0,-761.263133,-382.577826
1,-402.952336,-968.200039
2,-311.357237,-789.621571
3,-676.945966,-198.339994
4,754.711154,-681.695649


### Step 4: Using the customers_pca_df Dataframe, utilize the elbow method to determine the optimal value of k.

In [11]:
# Create a a list to store inertia values and the values of k
inertia = []
k = list(range(1, 11))

In [12]:
# Create a for-loop where each value of k is evaluated using the K-means algorithm
# Fit the model using the service_ratings DataFrame
# Append the value of the computed inertia from the `inertia_` attribute of the KMeans model instance
for i in k:
    k_model = KMeans(n_clusters=i, random_state=0)
    k_model.fit(customers_pca_df)
    inertia.append(k_model.inertia_)

In [13]:
# Define a DataFrame to hold the values for k and the corresponding inertia
elbow_data = {"k": k, "inertia": inertia}

# Create the DataFrame from the elbow data
df_elbow = pd.DataFrame(elbow_data)

# Review the DataFrame
df_elbow.head()

,k,inertia
0,1,1.570535e+11
1,2,1.024687e+11
2,3,5.865475e+10
3,4,4.333276e+10
4,5,3.552790e+10


In [14]:
# Plot the DataFrame
df_elbow.hvplot.line(
    x="k", 
    y="inertia", 
    title="Elbow Curve", 
    xticks=k
)

:Curve   [k]   (inertia)

### Step 5: Segment the `customers_pca_df`  DataFrame using the K-means algorithm.

In [15]:
# Define the model Kmeans model using the optimal value of k for the number of clusters.
model = KMeans(n_clusters=4, random_state=0)

# Fit the model
model.fit(customers_pca_df)

# Make predictions
k_3 = model.predict(customers_pca_df)

# Create a copy of the customers_pca_df DataFrame
customer_pca_predictions_df = customers_pca_df.copy()

# Add a class column with the labels
customer_pca_predictions_df["Report_District_segments"] = k_3

In [16]:
# Plot the clusters
customer_pca_predictions_df.hvplot.scatter(
    x="PCA1",
    y="PCA2",
    by="Report_District_segments"
)

:NdOverlay   [Report_District_segments]
   :Scatter   [PCA1]   (PCA2)

### Step 6: Segment the `customers_transformed_df` DataFrame with all factors using the K-means algorithm

In [19]:
# Define the model Kmeans model using k=3 clusters
model = KMeans(n_clusters=3, random_state=0)

# Fit the model
model.fit(customers_transformed_df)

# Make predictions
k_3 = model.predict(customers_transformed_df)

# Create a copy of the customers_transformed_df DataFrame
customers_transformed_predictions_df = customers_transformed_df.copy()

# Add a class column with the labels
customers_transformed_predictions_df["Report_District_segments"] = k_3

In [20]:
# Plot the clusters using the first two feature columns
customers_transformed_predictions_df.hvplot.scatter(
    x="TIME OCC",
    y="Rpt Dist No",
    by="Report_District_segments"
)

:NdOverlay   [Report_District_segments]
   :Scatter   [TIME OCC]   (Rpt Dist No)